# 🎨 Neural Gesture Mannequin — Interactive Dual-Mode Inference & Anime Sketch Generation
**Author:** Muhammad Aliffandy (Apple Developer Academy & PukulEnam AI/ML Engineer)  
**Model:** Multi-Task Volumetric PoseNet (Trained on 114,554 COCO Images on NVIDIA DGX H100)  

### 🌟 Complete End-to-End Pipeline:
1. **Input Photo:** Any custom human action photo (uploaded from your computer or URL).
2. **Mode 1: Skeleton Wireframe:** 17 kinematic nodes & Line of Action predicted by your DGX-trained model.
3. **Mode 2: Volumetric 3D Mannequin:** Tapered cylinders, cross-contour perspective ellipses & torso boxes.
4. **Mode 3: Low-Poly Anime Mannequin Sketch:** AI-synthesized faceted anime underdrawing line art directly guided by your model's skeleton rig!

## 🛠️ Step 1: Clone Repository & Install Dependencies

In [ ]:
# 1. Clone repository
!rm -rf hp-tra
!git clone https://github.com/MuhammadAliffandy/hp-tra.git
%cd hp-tra

# 2. Install lightweight dependencies
!pip install -q pycocotools torchvision opencv-python matplotlib Pillow

## ☁️ Step 2: Mount Google Drive & Load Trained Model (`.pth`)

In [ ]:
from google.colab import drive
import os
import torch
from src.model import MultiTaskVolumetricPoseNet

# Mount Drive
drive.mount('/content/drive')

# Model path in your Google Drive (transferred from DGX)
MODEL_PATH = "/content/drive/MyDrive/Neural_Gesture_Mannequin_Models/volumetric_posenet_best.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

# Instantiate Model (MobileNetV3-Large)
model = MultiTaskVolumetricPoseNet(backbone_name="mobilenet_v3_large", pretrained=False).to(device)

if os.path.exists(MODEL_PATH):
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        val_err = checkpoint.get("val_err", "N/A")
        val_pck = checkpoint.get("val_pck", "N/A")
        print(f"✅ Model successfully loaded from Drive!")
        print(f"   - Best Val Pixel Error : {val_err:.2f} px")
        print(f"   - Best Val PCK@0.1     : {val_pck:.1f} %")
    else:
        model.load_state_dict(checkpoint)
        print("✅ Model state dict loaded successfully!")
else:
    print(f"⚠️ Checkpoint not found at: {MODEL_PATH}")
    print("Please verify that rclone transfer to xxdrive:Neural_Gesture_Mannequin_Models/ has completed.")

model.eval()

## 📸 Step 3: Upload Your Photo (or Provide URL)

In [ ]:
from google.colab import files
from PIL import Image
import urllib.request

# Choose input mode:
# Option A: Upload from your laptop
# Option B: Use an image URL from the web

IMAGE_URL = ""  # Leave empty if you want to upload a file from your computer
TARGET_IMAGE_PATH = "test_input.jpg"

if IMAGE_URL.strip() != "":
    print(f"🌐 Downloading image from URL: {IMAGE_URL}")
    urllib.request.urlretrieve(IMAGE_URL, TARGET_IMAGE_PATH)
    print("✅ Download complete!")
else:
    print("📁 Click 'Choose Files' below to upload your photo:")
    uploaded = files.upload()
    if uploaded:
        TARGET_IMAGE_PATH = list(uploaded.keys())[0]
        print(f"✅ Uploaded: {TARGET_IMAGE_PATH}")
    else:
        print("No file selected, using default sample.")

## 🚀 Step 4: Run DGX Model Inference & Display 4-Panel Master Artist Canvas

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
from src.visualize import create_dual_mode_canvas, draw_skeleton_mode, draw_volume_mode

# 1. Preprocess Input Image
transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

raw_pil = Image.open(TARGET_IMAGE_PATH).convert("RGB")
resized_pil = raw_pil.resize((256, 256), Image.Resampling.BILINEAR)
raw_bgr = cv2.cvtColor(np.array(resized_pil), cv2.COLOR_RGB2BGR)

inp_tensor = transform(raw_pil).unsqueeze(0).to(device)

# 2. Forward Pass through Multi-Task Volumetric PoseNet
with torch.no_grad():
    pred_coords, pred_volume, pred_vis_logits = model(inp_tensor)
    
    coords_np = pred_coords[0].cpu().numpy()             # [17, 2] in [0, 1]
    volume_np = pred_volume[0].cpu().numpy()             # [6] in [0, 1]
    vis_np = torch.sigmoid(pred_vis_logits[0]).cpu().numpy() # [17] confidence

# 3. Render Individual Panels
blank_white = np.full((256, 256, 3), 255, dtype=np.uint8)

panel1_photo = np.array(resized_pil) # RGB
panel2_skeleton = cv2.cvtColor(draw_skeleton_mode(raw_bgr, coords_np, vis_np, conf_thresh=0.25), cv2.COLOR_BGR2RGB)
panel3_volume = cv2.cvtColor(draw_volume_mode(blank_white, coords_np, volume_np, vis_np, conf_thresh=0.25), cv2.COLOR_BGR2RGB)
panel4_combined = cv2.cvtColor(draw_volume_mode(blank_white, coords_np, volume_np, vis_np, conf_thresh=0.25), cv2.COLOR_BGR2RGB)
panel4_combined = cv2.cvtColor(draw_skeleton_mode(cv2.cvtColor(panel4_combined, cv2.COLOR_RGB2BGR), coords_np, vis_np, conf_thresh=0.25), cv2.COLOR_BGR2RGB)

# 4. Plot Master 4-Panel Comparison
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

axes[0].imshow(panel1_photo)
axes[0].set_title("[1] Input Photo", fontsize=13, fontweight='bold', pad=10)
axes[0].axis("off")

axes[1].imshow(panel2_skeleton)
axes[1].set_title("[2] Mode 1: Skeleton Wireframe\n(Line of Action & Kinematics)", fontsize=12, fontweight='bold', pad=10)
axes[1].axis("off")

axes[2].imshow(panel3_volume)
axes[2].set_title("[3] Mode 2: 3D Volumetric Mannequin\n(Cylinders & Cross-Contours)", fontsize=12, fontweight='bold', pad=10)
axes[2].axis("off")

axes[3].imshow(panel4_combined)
axes[3].set_title("[4] Combined Artist Canvas\n(Rig + Volumetric Primitives)", fontsize=12, fontweight='bold', pad=10)
axes[3].axis("off")

plt.tight_layout()
output_preview_path = "master_mannequin_preview.png"
plt.savefig(output_preview_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"🎉 SUCCESS! High-res comparison saved to: {output_preview_path}")

## 🎨 Step 5: Generate Clean Anime Mannequin Pose Base (Atari / アタリ Underdrawing)
Generates a **Single Character (Solo)** clean **2D Anime Drawing Base / Anatomy Mannequin** directly from your model's skeleton rig! Exactly matches the clean anime reference base aesthetic: smooth contour lines, blank anime head with facial crosshair guidelines, and clean flat background.

In [ ]:
# ==============================================================================
# 🎨 STEP 5: CLEAN 2D ANIME MANNEQUIN POSE BASE (SINGLE CHARACTER SOLO)
# ==============================================================================
# 1. Install generative libraries
!pip install -q diffusers transformers accelerate controlnet-aux

import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from PIL import Image
import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.visualize import draw_skeleton_mode

print("⏳ Loading Pretrained ControlNet-OpenPose & Anime Engine...")

# 2. Load Pretrained ControlNet OpenPose
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_openpose",
    torch_dtype=torch.float16
)

# 3. Load Anime Base Engine (AnyLoRA Anime)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "Lykon/AnyLoRA",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()

print("✅ 2D Anime Pose Base Engine Ready!")

# 4. Prepare Standard Clean Black Canvas Skeleton for ControlNet
blank_black = np.zeros((256, 256, 3), dtype=np.uint8)
control_skeleton_bgr = draw_skeleton_mode(blank_black, coords_np, vis_np, conf_thresh=0.25)
control_skeleton_rgb = cv2.cvtColor(control_skeleton_bgr, cv2.COLOR_BGR2RGB)
skeleton_for_control = Image.fromarray(control_skeleton_rgb).resize((512, 512), Image.Resampling.NEAREST)

# 5. Prompt Engineering for Clean 2D Anime Mannequin Base (Exact Match to Your Reference!)
# Enforces: Single Character (solo), clean digital lines, blank face with crosshair, hairless/bald, no clothes, no shading
GENDER = "1girl"  # Choose "1girl" or "1boy"

prompt = (
    f"masterpiece, best quality, (solo:1.5), ({GENDER}:1.3), single character, "
    "(anime pose base:1.5), (drawing reference mannequin:1.4), (atari:1.3), "
    "(clean smooth line art:1.4), brown lineart, monochrome outline, minimal lines, "
    "(blank featureless face:1.4), (facial crosshair guideline:1.4), egg shaped anime head, "
    "hairless, bald, no hair, unclothed, simple female anatomy mannequin, "
    "smooth digital body contour, collarbone line, hip line, knee joint circles, "
    "flat pure white background, digital artist reference sheet, 2d vector style"
)

negative_prompt = (
    "(multiple characters:1.6), (2girls:1.6), (2boys:1.6), (crowd:1.6), (group:1.6), "
    "(hair:1.5), (clothes:1.5), (clothing:1.5), dress, underwear, bra, panties, "
    "(detailed eyes:1.5), (pupils:1.5), (mouth:1.4), (nose:1.3), detailed face, "
    "(shading:1.4), (shadows:1.4), (gradients:1.4), (3d render:1.4), (render:1.3), "
    "(photorealistic:1.5), (realistic skin:1.5), (colors:1.4), colorful, texture, "
    "(faceted:1.4), (polygons:1.4), (wooden mannequin:1.4), (wood:1.3), "
    "(messy lines:1.4), (charcoal:1.4), (crosshatching:1.4), pencil grain, dirty, noise, "
    "dark background, grey background, floor, wall, furniture, interior"
)

# 6. Generate Clean Anime Mannequin Line Art (~3-4 seconds on Colab GPU!)
print("🎨 Synthesizing Single Character Anime Pose Mannequin Base...")
generator = torch.manual_seed(42)

anime_sketch = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=skeleton_for_control,
    width=512,
    height=512,
    num_inference_steps=24,
    guidance_scale=6.5,
    controlnet_conditioning_scale=1.1, # Strictly locks the pose to your DGX skeleton!
    generator=generator
).images[0]

# 7. Plot 3-Panel Master Showcase
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(panel1_photo)
axes[0].set_title("[1] Input Photo", fontsize=13, fontweight='bold', pad=10)
axes[0].axis("off")

axes[1].imshow(control_skeleton_rgb)
axes[1].set_title("[2] DGX Multi-Task Skeleton Rig", fontsize=13, fontweight='bold', pad=10)
axes[1].axis("off")

axes[2].imshow(anime_sketch)
axes[2].set_title("[3] Output: 2D Anime Pose Base (Atari Mannequin)", fontsize=13, fontweight='bold', pad=10)
axes[2].axis("off")

plt.tight_layout()
output_anime_path = "anime_mannequin_base_sketch.png"
anime_sketch.save(output_anime_path)
plt.savefig("master_pipeline_showcase.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"🎉 SUCCESS! Clean 2D Anime Mannequin Base saved to: {output_anime_path}")
print(f"🖼️ Full pipeline showcase saved to: master_pipeline_showcase.png")
